# Prever eventos de anomalias nos datasets de taxi

***Deteção de anomalias não supervisionadas em dados do tipo timeseries com algoritmos de Random Cut Forest.***

# Introdução
***

Amazon SageMaker Random Cut Forest (RCF) é um algoritmo para detectar anomalias em dataset. Examples: a) detectar uso do site fora do padrao, b) detectar sensores que estão fora do padrão, ou c) quando mudanças no transito refletem em ocorrencias especiais. 

Neste dataset do período de 6 meses identificaremos anomalias como a maratona de NYC, Ações de Graças, Natal, Ano novo, ou dias de nevasca de neve.

In [ ]:
import boto3
import botocore
import sagemaker
import sys


bucket = 'alexbc.datalake'
prefix = 'machine-learning-models/sagemaker/rcf-benchmarks'
execution_role = sagemaker.get_execution_role()

boto3.Session().client('s3').head_bucket(Bucket=bucket)
print('Training input/output sera armazenado no s3://{}/{}'.format(bucket, prefix))

In [ ]:
%%time

import pandas as pd
import urllib.request

data_filename = 'passengers_by_halfhour.csv'
data_source = 'https://s3.amazonaws.com/alexbc.datalake/ny-taxi-data/taxi_datamart/' + data_filename
urllib.request.urlretrieve(data_source, data_filename)
taxi_data = pd.read_csv(data_filename, delimiter=',')

## Análise exploratória

In [ ]:
taxi_data.head()

In [ ]:
%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 100

taxi_data.plot()

In [ ]:
taxi_data[5952:6000]

## Converter o formato para um formato aceitável pelo Random Cut Forest (RecordIO)

In [ ]:
def convert_and_upload_training_data(ndarray, bucket, prefix, filename='data.pbr'):
    import boto3
    import os
    from sagemaker.amazon.common import numpy_to_record_serializer
    
    # convert numpy array to Protobuf RecordIO format
    serializer = numpy_to_record_serializer()
    buffer = serializer(ndarray)
    
    # Upload to S3
    s3_object = os.path.join(prefix, 'train', filename)
    boto3.Session().resource('s3').Bucket(bucket).Object(s3_object).upload_fileobj(buffer)
    
    s3_path = 's3://{}/{}'.format(bucket, s3_object)
    return s3_path


# RCV only works on an array of values.
s3_train_data = convert_and_upload_training_data(
    taxi_data.value.as_matrix().reshape(-1,1),
    bucket,
    prefix)
print('Uploaded para {}'.format(s3_train_data))

# Treino

***


In [ ]:
import boto3

# select the algorithm container based on this notebook's current location
containers = {'us-west-2': '174872318107.dkr.ecr.us-west-2.amazonaws.com/randomcutforest:latest',
              'us-east-1': '382416733822.dkr.ecr.us-east-1.amazonaws.com/randomcutforest:latest',
              'us-east-2': '404615174143.dkr.ecr.us-east-2.amazonaws.com/randomcutforest:latest',
              'eu-west-1': '438346466558.dkr.ecr.eu-west-1.amazonaws.com/randomcutforest:latest',
              'ap-northeast-1': '351501993468.dkr.ecr.ap-northeast-1.amazonaws.com/randomcutforest:latest'}
region_name = boto3.Session().region_name
container = containers[region_name]

print('Container do RCF que esta sendo utilizado: {} ({})'.format(container, region_name))

## Hyperparametros

In [ ]:
session = sagemaker.Session()

# informações gerais do job
rcf = sagemaker.estimator.Estimator(
    container,
    execution_role,
    output_path='s3://{}/{}/output'.format(bucket, prefix),
    train_instance_count=1,
    train_instance_type='ml.m4.xlarge',
    sagemaker_session=session,
)

# principais hyperparameterer como numero de amostras da arvore, numero da arvore
rcf.set_hyperparameters(
    num_samples_per_tree = 200,
    num_trees = 50,
    feature_dim = 1,
)

s3_train_input = sagemaker.session.s3_input(
    s3_train_data,
    distribution='ShardedByS3Key',
    content_type='application/x-recordio-protobuf',
)

# run the training job on input data stored in S3
rcf.fit({'train': s3_train_input})

In [ ]:
print('Nome do job de treino: {}'.format(rcf.latest_training_job.job_name))

# Deploy da infra dos modelos para Inferencia

In [ ]:
rcf_inference = rcf.deploy(
    initial_instance_count=1,
    instance_type='ml.m4.xlarge',
)

In [ ]:
print('nome do Endpoint: {}'.format(rcf_inference.endpoint))

## Vamos pegar um amostra de dados para ver os scores

In [ ]:
from sagemaker.predictor import csv_serializer, json_deserializer

rcf_inference.content_type = 'text/csv'
rcf_inference.serializer = csv_serializer
rcf_inference.deserializer = json_deserializer

In [ ]:
taxi_data_numpy = taxi_data.value.as_matrix().reshape(-1,1)
results = rcf_inference.predict(taxi_data_numpy[:6])

print(results)

## Atribuindo Scores para anomalias no dataset

In [ ]:
results = rcf_inference.predict(taxi_data_numpy)
scores = [datum['score'] for datum in results['scores']]

# add scores to taxi data frame and print first few values
taxi_data['score'] = pd.Series(scores, index=taxi_data.index)
taxi_data.head()

In [ ]:
#fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

#
# *Try this out* - change `start` and `end` to zoom in on the 
# anomaly found earlier in this notebook
#
start, end = 0, len(taxi_data)
start, end = 5500, 6500
taxi_data_subset = taxi_data[start:end]

ax1.plot(taxi_data_subset['value'], color='C0', alpha=0.8)
ax2.plot(taxi_data_subset['score'], color='C1')

ax1.grid(which='major', axis='both')

ax1.set_ylabel('Taxi Ridership', color='C0')
ax2.set_ylabel('Anomaly Score', color='C1')

ax1.tick_params('y', colors='C0')
ax2.tick_params('y', colors='C1')

ax1.set_ylim(0, 40000)
ax2.set_ylim(min(scores), 1.4*max(scores))
fig.set_figwidth(10)

In [ ]:
score_mean = taxi_data['score'].mean()
score_std = taxi_data['score'].std()
score_cutoff = score_mean + 3*score_std

anomalies = taxi_data_subset[taxi_data_subset['score'] > score_cutoff]
anomalies


* `2014-11-02` - NYC Marathon
* `2015-01-01` - New Year's Eve
* `2015-01-27` - Snowstorm

In [ ]:
ax2.plot(anomalies.index, anomalies.score, 'ko')
fig

## Deletar endpoint

In [ ]:
sagemaker.Session().delete_endpoint(rcf_inference.endpoint)